# Project 2 - Mobile Robot Corridor

This walkthrough compares a weak baseline with predictive constrained tracking for a mobile robot moving through a narrowed and laterally shifted corridor. The low-level MPC construction and plot formatting stay local to the project; the scenario, model, limits, and conclusions are visible here.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys
import numpy as np
from IPython.display import FileLink, display

repo_root = Path.cwd()
while not (repo_root / "projects").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from projects.project_2_mobile_robot_corridor import config, scenario
from projects.project_2_mobile_robot_corridor.animation import save_corridor_animation
from projects.project_2_mobile_robot_corridor.plots import (
    plot_horizon_comparison,
    plot_input,
    plot_lateral_error,
    plot_margin,
    plot_top_view,
)
from projects._shared.mobile_robot import tracking_error_matrices

output_root = Path(os.environ.get("THIMPC_OUTPUT_DIR", "/tmp/thimpc_walkthroughs"))
project_output = output_root / "project_2_mobile_robot_corridor"
np.set_printoptions(precision=3, suppress=True)


## Main Experiment Parameters

The key numbers are explicit here. The project module still supplies defaults, but the notebook should be readable on its own.


In [ ]:
dt = config.DT
horizon = config.HORIZON
n_steps = int(os.environ.get("THIMPC_PROJECT2_STEPS", str(config.STEPS)))
v_ref = config.V_REF
omega_ref = config.OMEGA_REF
radius = config.RADIUS
initial_error = config.INITIAL_ERROR.copy()
input_limits = (config.DELTA_MIN[0], config.DELTA_MAX[0])
input_rate_limit = config.DELTA_RATE[0]
corridor_tightening = 0.0
yaw_rate_bias = 0.0

print(f"dt = {dt}, horizon = {horizon}, simulated steps = {n_steps}")
print(f"reference speed = {v_ref}, reference yaw rate = {omega_ref}, radius = {radius}")
print("initial tracking error [xi, eta, psi] =", initial_error)
print("yaw-rate correction limits =", input_limits)
print("yaw-rate correction rate limit =", input_rate_limit)
print("narrow corridor center [deg] =", np.degrees(config.NARROW_CENTER))
print("narrow corridor angular width [deg] =", np.degrees(config.NARROW_WIDTH))
print("narrow corridor eta offset =", scenario.NARROW_OFFSET)
print("MPC slack weight =", config.MPC_SLACK_WEIGHT)


## Local Error Model And Corridor Constraints

The controller uses a local tracking-error model around the circular reference:

`e[k+1] = A_e e[k] + B_e delta_omega[k]`

where `e = [xi, eta, psi]` contains longitudinal error, lateral error, and heading error.

The corridor mainly constrains `eta`. Around the critical section, the allowed lateral-error interval smoothly narrows and shifts inward. The nominal reference has `eta = 0`, and near the center of the narrowed section that value is deliberately outside the allowed state set.


In [ ]:
A_error, B_error = tracking_error_matrices(dt, v_ref, omega_ref)
wide_phi = np.array([0.0])
narrow_phi = np.array([config.NARROW_CENTER])
wide_bounds = scenario.corridor_error_bounds(wide_phi, margin=corridor_tightening)
narrow_bounds = scenario.corridor_error_bounds(narrow_phi, margin=corridor_tightening)

print("A_error =")
print(A_error)
print("B_error =")
print(B_error)
print("wide corridor eta bounds =", float(wide_bounds[0][0]), float(wide_bounds[1][0]))
print("critical corridor eta bounds =", float(narrow_bounds[0][0]), float(narrow_bounds[1][0]))
print("is desired path eta = 0 feasible at the critical section?", bool(narrow_bounds[0][0] <= 0.0 <= narrow_bounds[1][0]))
print("model mismatch enters later as yaw_rate_bias in the plant rollout")


## Baseline, Saturated LQR, And MPC

What are we comparing?
- `baseline`: a low-gain local tracker with the same input and rate limits, but no look-ahead constraints;
- `saturated LQR`: local LQR feedback with input and input-rate clipping;
- `hard MPC diagnostic`: hard corridor bounds, useful for diagnosing true infeasibility when the realized state is already outside the admissible set;
- `MPC`: predicts the shifted future corridor and uses heavily priced slack for robust recovery near numerical or setup infeasibility.


In [ ]:
baseline = scenario.simulate_tracker("baseline", n_steps)
saturated = scenario.simulate_tracker("saturated_lqr", n_steps)
hard_mpc = scenario.simulate_tracker("mpc", n_steps, horizon=horizon, margin=corridor_tightening)
mpc = scenario.simulate_tracker("mpc", n_steps, horizon=horizon, margin=corridor_tightening, soft_corridor=True)

runs = {
    "baseline": baseline,
    "saturated LQR": saturated,
    "MPC": mpc,
}

time = np.arange(n_steps + 1) * dt
plot_top_view(project_output / "figures" / "top_view.png", mpc["reference"][:, :2], runs, radius)
plot_lateral_error(project_output / "figures" / "lateral_error.png", time, runs)
plot_input(project_output / "figures" / "yaw_rate_correction.png", time, runs, input_limits[0], input_limits[1])
plot_margin(project_output / "figures" / "corridor_margin.png", time, runs)


## Replay: Moving Through The Corridor

Run this cell after the simulation to generate a replay.
The replay is saved under outputs/ and is not committed.
Open the generated GIF file from the displayed link.


In [ ]:
replay_output = repo_root / "outputs" / "project_2_mobile_robot_corridor"
replay_output.mkdir(parents=True, exist_ok=True)
replay_path = replay_output / "corridor_comparison.gif"

animation_stride = max(1, n_steps // 45)
save_corridor_animation(
    replay_path,
    runs,
    radius=radius,
    stride=animation_stride,
    interval=80,
    fps=12,
)

print(f"Replay saved to: {replay_path}")
display(FileLink(replay_path))


### Interpretation

- The baseline is a plausible local tracker, but it does not know that the reference will become infeasible in the narrowed section.
- Saturated LQR usually improves the local response, but clipping still does not make it predictive.
- MPC uses the shifted corridor bounds inside the horizon, so it moves before the narrow section becomes critical.
- The hard MPC run is a diagnostic: if the current measured state already violates a hard state bound, the optimization problem can be truly infeasible.
- The margin plot is the safety-relevant plot: negative means corridor violation.


## Metrics Comparison

The plots show the story; the metrics make it auditable.


In [ ]:
for label, run in {**runs, "hard MPC diagnostic": hard_mpc}.items():
    summary = scenario.summarize_run(run)
    print(
        f"{label:20s} "
        f"min margin = {summary['minimum_margin']:+.3f}, "
        f"max violation = {summary['maximum_violation']:.3f}, "
        f"violating samples = {summary['corridor_violation_count']:3d}, "
        f"rms pos = {summary['rms_position_error']:.3f}, "
        f"rms eta = {summary['rms_lateral_error']:.3f}, "
        f"max |delta| = {summary['max_abs_input']:.3f}, "
        f"failures = {summary['solver_failures']}"
    )
    if summary["status_counts"]:
        print("  statuses:", summary["status_counts"])


### Interpretation

- A smaller or zero violation count is the main constrained-control claim.
- Input limits should be read together with the margin plot; a controller may be saturated exactly where the corridor narrows.
- RMS error alone is not enough: a low RMS can still hide a short safety violation.


## Horizon Length And Model Mismatch

What should students modify?
- shorten the horizon and watch the narrowed corridor become harder;
- add yaw-rate bias to represent plant/model mismatch;
- tighten the corridor margin to create a more conservative controller.


In [ ]:
horizon_runs = {
    5: scenario.simulate_tracker("mpc", n_steps, horizon=5, soft_corridor=True),
    15: scenario.simulate_tracker("mpc", n_steps, horizon=15, soft_corridor=True),
    30: scenario.simulate_tracker("mpc", n_steps, horizon=30, soft_corridor=True),
}
plot_horizon_comparison(project_output / "figures" / "horizon_comparison.png", time, horizon_runs)

biased_soft_mpc = scenario.simulate_tracker("mpc", n_steps, horizon=horizon, soft_corridor=True, yaw_rate_bias=0.035)
tightened_soft_mpc = scenario.simulate_tracker("mpc", n_steps, horizon=horizon, margin=0.08, soft_corridor=True)

for label, run in {"soft MPC with yaw bias": biased_soft_mpc, "tightened soft MPC": tightened_soft_mpc}.items():
    summary = scenario.summarize_run(run)
    print(f"{label:24s} min margin = {summary['minimum_margin']:+.3f}, total slack = {summary['total_slack']:.3f}")


### Interpretation

- A short horizon may see the narrow section too late.
- Tightening gives the optimizer a buffer, but it can also make the problem harder.
- Model mismatch can reduce the real margin even when the nominal prediction looked safe.


## Modification Cell

Change one value and rerun the comparison.

Suggested experiments:
- increase `modified_corridor_tightening` to narrow the usable corridor;
- decrease `modified_horizon` to make the controller more short-sighted;
- increase `modified_yaw_rate_bias` to test mismatch.


In [ ]:
# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.
